In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import weight_norm
from torch.nn.init import kaiming_normal_

## Basic Blocks

In [ ]:
# === Activation Dictionary ===
activation_dict = {
    'relu': F.relu,
    'leaky_relu': F.leaky_relu,
    'tanh': F.tanh,
    'sigmoid': F.sigmoid,
    'gelu': F.gelu
}

# === CBAM Attention ===
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.shared_mlp = nn.Sequential(
            nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.shared_mlp(self.avg_pool(x))
        max_out = self.shared_mlp(self.max_pool(x))
        out = avg_out + max_out
        return self.sigmoid(out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv(x_cat))

class CBAM(nn.Module):
    def __init__(self, channels, reduction_ratio=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.channel_attention = ChannelAttention(channels, ratio=reduction_ratio)
        self.spatial_attention = SpatialAttention(kernel_size)

    def forward(self, x):
        out = x * self.channel_attention(x)
        out = out * self.spatial_attention(out)
        return out

# === FiLM Layer ===
class FiLM(nn.Module):
    def __init__(self, in_channels, cond_dim):
        super(FiLM, self).__init__()
        self.gamma = nn.Linear(cond_dim, in_channels)
        self.beta = nn.Linear(cond_dim, in_channels)

    def forward(self, x, cond):
        gamma = self.gamma(cond).unsqueeze(2).unsqueeze(3)
        beta = self.beta(cond).unsqueeze(2).unsqueeze(3)
        return gamma * x + beta



# --- Utility Normalization ---
def get_norm(num_channels, normalization_type='batchnorm'):
    if normalization_type == 'groupnorm':
        num_groups = min(8, num_channels)
        while num_channels % num_groups != 0 and num_groups > 1:
            num_groups -= 1
        return nn.GroupNorm(num_groups, num_channels)
    elif normalization_type == 'batchnorm':
        return nn.BatchNorm2d(num_channels)
    else:
        raise ValueError(f"Unknown normalization type: {normalization_type}")


class ResBlock(nn.Module):
    def __init__(self, channels, cond_dim, activation_fn=F.gelu, normalization_type='batchnorm'):
        super().__init__()
        self.normalization_type = normalization_type

        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = get_norm(channels, normalization_type)
        self.film1 = FiLM(channels, cond_dim)

        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = get_norm(channels, normalization_type)
        self.film2 = FiLM(channels, cond_dim)

        self.activation_fn = activation_fn
        self._initialize_weights()

    def _initialize_weights(self):
        kaiming_normal_(self.conv1.weight, mode='fan_out', nonlinearity='relu')
        kaiming_normal_(self.conv2.weight, mode='fan_out', nonlinearity='relu')

    def forward(self, x, cond):
        residual = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.film1(out, cond)
        out = self.activation_fn(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.film2(out, cond)

        out += residual
        out = self.activation_fn(out)
        return out

# === Residual Label Embedding ===
class ResidualLabelEmbedding(nn.Module):
    def __init__(self, num_classes, embedding_dim=128, hidden_dims=[16, 32, 64]):
        super().__init__()
        self.fc1 = nn.Linear(num_classes, hidden_dims[0])
        self.fc2 = nn.Linear(hidden_dims[0], hidden_dims[1])
        self.fc3 = nn.Linear(hidden_dims[1], hidden_dims[2])
        self.fc4 = nn.Linear(hidden_dims[2], embedding_dim)
        self.act = F.gelu
        
        self.proj1 = nn.Linear(hidden_dims[0], hidden_dims[1])
        self.proj2 = nn.Linear(hidden_dims[1], hidden_dims[2])

    def forward(self, y):
        out1 = self.act(self.fc1(y))
        out2 = self.act(self.fc2(out1) + self.proj1(out1))
        out3 = self.act(self.fc3(out2) + self.proj2(out2))
        y_embedding = self.act(self.fc4(out3))
        return y_embedding

## Coupling Layer

In [ ]:
# === Coupling Layer ===
class Coupling(nn.Module):
    def __init__(self, input_shape=(4, 91, 91), num_classes=3, reg=0.01, activation_fn=F.gelu,
                 normalization_type='groupnorm', filters=(32, 64, 128), verbose=False):
        super().__init__()
        self.input_shape = input_shape
        self.reg = reg
        self.activation_fn = activation_fn
        self.normalization_type = normalization_type
        self.verbose = verbose

        filter_1, filter_2, filter_3 = filters

        # Residual label embedding
        self.label_embedding = ResidualLabelEmbedding(num_classes, embedding_dim=128)

        # Convolutional layers with weight normalization
        self.conv1 = weight_norm(nn.Conv2d(input_shape[0], filter_1, 5, stride=2, padding=2))
        self.bn1 = self.get_normalization(filter_1)
        self.film1 = FiLM(filter_1, 128)

        self.conv2 = weight_norm(nn.Conv2d(filter_1, filter_2, 3, stride=2, padding=1))
        self.bn2 = self.get_normalization(filter_2)
        self.film2 = FiLM(filter_2, 128)
        self.cbam2 = CBAM(filter_2)

        self.conv3 = weight_norm(nn.Conv2d(filter_2, filter_3, 3, stride=2, padding=1))
        self.bn3 = self.get_normalization(filter_3)
        self.film3 = FiLM(filter_3, 128)
        self.cbam3 = CBAM(filter_3)

        # Residual blocks
        self.resblock1 = ResBlock(filter_2, 128, activation_fn, normalization_type='groupnorm')
        self.resblock2 = ResBlock(filter_3, 128, activation_fn, normalization_type='groupnorm')

        self.dropout = nn.Dropout(0.3)

        # Flattened feature size calculation
        def conv_output_size(H_in, kernel, stride, padding):
            return (H_in + 2*padding - kernel) // stride + 1

        H_out1 = conv_output_size(input_shape[1], 5, 2, 2)
        H_out2 = conv_output_size(H_out1, 3, 2, 1)
        H_out3 = conv_output_size(H_out2, 3, 2, 1)
        flattened_size = filter_3 * H_out3 * H_out3

        # Fully connected layers for translation (t)
        self.t_fc1 = nn.Linear(flattened_size, 64)
        self.t_fc2 = nn.Linear(128, 64)
        self.t_fc3 = nn.Linear(64 + 64, input_shape[0] * input_shape[1] * input_shape[2])

        # Fully connected layers for scale (s)
        self.s_fc1 = nn.Linear(flattened_size, 64)
        self.s_fc2 = nn.Linear(128, 64)
        self.s_fc3 = nn.Linear(64 + 64, input_shape[0] * input_shape[1] * input_shape[2])

    def get_normalization(self, num_features):
        if self.normalization_type == 'groupnorm':
            return nn.GroupNorm(8, num_features)
        elif self.normalization_type == 'batchnorm':
            return nn.BatchNorm2d(num_features)
        else:
            raise ValueError(f"Unknown normalization type: {self.normalization_type}")

    def _log_shape(self, name, tensor):
        if self.verbose:
            print(f"{name}: {list(tensor.shape)}")

    def forward(self, x, y):
        # === Residual label embedding ===
        y_embedding = self.label_embedding(y)
        self._log_shape("Label embedding", y_embedding)

        # === Convolutional backbone ===
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.film1(x, y_embedding)
        x = self.activation_fn(x)
        x = self.dropout(x)
        self._log_shape("After conv1", x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.film2(x, y_embedding)
        x = self.activation_fn(x)
        x = self.cbam2(x)
        x = self.resblock1(x, y_embedding)
        x = self.dropout(x)
        self._log_shape("After conv2 + resblock1", x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = self.film3(x, y_embedding)
        x = self.activation_fn(x)
        x = self.cbam3(x)
        x = self.resblock2(x, y_embedding)
        x = self.dropout(x)
        self._log_shape("After conv3 + resblock2", x)

        # Flatten
        x_flat = x.view(x.shape[0], -1)
        self._log_shape("After flattening", x_flat)

        # === Translation (t) ===
        t = self.activation_fn(self.t_fc1(x_flat))
        t_emb = self.activation_fn(self.t_fc2(y_embedding))
        t = torch.cat((t, t_emb), dim=1)
        t = self.t_fc3(t).view(-1, *self.input_shape)
        self._log_shape("t (translation)", t)

        # === Scale (s) ===
        s = self.activation_fn(self.s_fc1(x_flat))
        s_emb = self.activation_fn(self.s_fc2(y_embedding))
        s = torch.cat((s, s_emb), dim=1)
        s = 0.5 * torch.tanh(self.s_fc3(s)).view(-1, *self.input_shape)
        self._log_shape("s (scale)", s)

        return s, t